# Module 4 · Project — Knowledge Assistant v1

**From 0 to Agentic AI — DataHack Summit 2026**

🧩 **The build begins for real.** We assemble the **Knowledge Assistant** as an explicit
**LangGraph `StateGraph`** — the same reason→act→observe loop from Module 3, but now with
state, routing, and looping as first-class parts we control.

> 📝 **Your turn.** Cells marked **TODO** have the body removed — implement them. Run each section as you go. Stuck? Peek at the `_SOLUTION` notebook.

### What you'll build
1. A **State** that carries the message history
2. A **model node** that calls the LLM (with tools bound)
3. A **tool node** that runs any tool the model requested
4. A **conditional edge** — tools? loop back : finish
5. `compile()`, visualize, and **run** the assistant

---
## Setup

In [ ]:
# Install the workshop stack (Colab). Locally, use `uv sync` instead.
# Version ranges match src/pyproject.toml (the single source of truth).
!pip install -q "langchain>=1.2,<2" "langchain-openai>=1.1,<2" \
               "langgraph>=1.0,<2" "langchain-tavily>=0.2"

In [ ]:
import os
from getpass import getpass

# Local: load keys from src/.env (walks up to find it). Colab: prompts for missing keys.
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    pass

for key in ["OPENAI_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")

---
## Step 1 · Tools

Our assistant starts with two tools (same idea as Module 3): a **web search** and a small
**internal directory** lookup. Later modules add real RAG, Slack, GitHub.

In [ ]:
from langchain_core.tools import tool
from langchain_tavily import TavilySearch

web_search = TavilySearch(max_results=3)

@tool
def lookup_employee(name: str) -> str:
    """Look up which team an employee works on, by first name."""
    directory = {"alessandro": "Billing", "sam": "Platform", "dana": "Data"}
    return directory.get(name.lower().strip(), "not found")

tools = [web_search, lookup_employee]

---
## Step 2 · The State

**TODO:** define the graph state. It needs a single field, `messages`, that **accumulates**
(use the `add_messages` reducer) rather than overwrites.

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages

class State(TypedDict):
    # TODO: one field `messages`, a list, annotated with the add_messages reducer
    ...

---
## Step 3 · The model node

Bind the tools to the LLM, then write the **agent node**: it takes the state, calls the
model on the message history, and returns the reply as a state update.

**TODO:** implement `call_model` so it returns `{"messages": [<the model reply>]}`.

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4.1-mini", model_provider="openai", temperature=0)
llm_with_tools = llm.bind_tools(tools)

def call_model(state: State):
    # TODO: invoke llm_with_tools on the messages, return them as a state update
    ...

---
## Step 4 · The tool node

When the model requests a tool, something has to run it. LangGraph ships a prebuilt
`ToolNode` that does exactly this — give it our list of tools.

In [ ]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)

---
## Step 5 · The conditional edge

After the model runs, we branch: **did it request a tool?** If yes → go to the tool node;
if no → we're done.

**TODO:** implement `should_continue` to return `"tools"` when the last message has
`tool_calls`, else `"end"`.

In [ ]:
def should_continue(state: State) -> str:
    # TODO: look at the last message; if it has .tool_calls return 'tools', else 'end'
    ...

---
## Step 6 · Wire the graph

Now connect the pieces into the loop:
`START → agent`, then the conditional edge (`agent → tools` or `agent → END`), and
crucially **`tools → agent`** so results feed back for another round.

**TODO:** add the nodes and edges, then compile.

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(State)
# TODO: add nodes 'agent' (call_model) and 'tools' (tool_node)
# TODO: START -> agent
# TODO: conditional edge from 'agent' via should_continue -> {'tools': 'tools', 'end': END}
# TODO: edge 'tools' -> 'agent' (the loop-back!)

assistant = builder.compile()

### Visualize the graph
See the loop you just built — note the edge from `tools` back to `agent`.

In [ ]:
from IPython.display import Image, display

display(Image(assistant.get_graph().draw_mermaid_png()))

---
## Step 7 · Run it

A small helper, then three questions: one that needs the directory, one that needs web
search, and one that needs neither (the model just answers).

In [ ]:
def ask(question: str):
    out = assistant.invoke({"messages": [("user", question)]})
    return out["messages"][-1].content

print(ask("What team is Dana on?"))

In [ ]:
print(ask("What is LangGraph, and what is its latest major version?"))

In [ ]:
print(ask("In one sentence, what is an AI agent?"))

### Inspect the full trace
Watch the loop: agent → (tool request) → tools → agent → final answer.

In [ ]:
out = assistant.invoke({"messages": [("user", "What team is Sam on?")]})
for m in out["messages"]:
    m.pretty_print()

---
## Key takeaways
- The Module 3 loop is now an explicit **graph**: `agent ⇄ tools`, with a conditional exit.
- **`ToolNode`** runs requested tools; **`should_continue`** is the routing decision.
- The **`tools → agent`** edge is what makes it iterate — the loop, made visible.

➡️ **Next (Module 5):** give this assistant real, reliable tools — web search, Slack
drafts, GitHub issues — and the design principles that keep them dependable.